In [0]:
%sql
WITH typed_sales AS (
    SELECT
        sale_id,
        source_file,
        quantity AS raw_quantity,
        TRY_CAST(quantity AS INT) AS quantity
    FROM ecom_external_catalog.retail_sales_dev.sales_bronze
)
SELECT
    sale_id,
    source_file,
    raw_quantity,
    quantity,
    CASE
        WHEN quantity IS NULL THEN 'INVALID_QUANTITY'
        WHEN quantity <= 0 THEN 'NON_POSITIVE_QUANTITY'
        ELSE 'VALID'
    END AS validation_status
FROM typed_sales
ORDER BY sale_id, source_file;

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

bronze = spark.table("ecom_external_catalog.retail_sales_dev.sales_bronze")

typed_sales = (
    bronze
    .withColumn("sale_date", F.expr("try_cast(sale_date AS DATE)"))
    .withColumn("quantity", F.expr("try_cast(quantity AS INT)"))
    .withColumn("unit_price", F.expr("try_cast(unit_price AS DECIMAL(12,2))"))
)

valid_sales = typed_sales.filter(F.col("quantity") > 0)

latest_per_sale = Window.partitionBy("sale_id").orderBy(
    F.col("ingested_at").desc(),
    F.col("source_file").desc(),
)

silver_preview = (
    valid_sales
    .withColumn("row_num", F.row_number().over(latest_per_sale))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

silver_preview.printSchema()
display(silver_preview.orderBy("sale_id"))

In [0]:
from delta.tables import DeltaTable

silver_table = "ecom_external_catalog.retail_sales_dev.sales_silver"

(
    DeltaTable.createIfNotExists(spark)
    .tableName(silver_table)
    .addColumns(silver_preview.schema)
    .execute()
)

target = DeltaTable.forName(spark, silver_table)

(
    target.alias("target")
    .merge(
        silver_preview.alias("source"),
        "target.sale_id = source.sale_id",
    )
    .whenMatchedUpdateAll(
        condition="""
            source.ingested_at > target.ingested_at
            OR (
                source.ingested_at = target.ingested_at
                AND source.source_file > target.source_file
            )
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

silver = spark.table(silver_table)
print(f"Silver row count: {silver.count()}")
display(silver.orderBy("sale_id"))